# IESO Coincident Peak Prediction — Backtesting & Financial Valuation

This notebook simulates operational deployment of the peak prediction model across
multiple historical base periods and quantifies the financial value for a
hypothetical 10 MW Class A customer.

**Key questions:**
- How accurately would the model have predicted peak days?
- How often would actual peaks fall within the predicted 3-hour window?
- What is the net financial value after accounting for false alarm costs?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
from pathlib import Path
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, mean_squared_error, r2_score)
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

PROJECT_ROOT = Path(r'C:/wamp64/www/Spec_Driven_Dev_Website')
DATA_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'data'
MODEL_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'models'

print('Libraries loaded successfully')

In [ ]:
# Load data and model
features = pd.read_parquet(DATA_DIR / 'ieso_features_daily.parquet')
features['Date'] = pd.to_datetime(features['Date'])
peaks = pd.read_parquet(DATA_DIR / 'ieso_peak_labels.parquet')
peaks['date'] = pd.to_datetime(peaks['date'])

model_artifact = joblib.load(MODEL_DIR / 'ieso_peak_model.joblib')
FEATURE_COLS = model_artifact['feature_columns']

print(f'Features: {len(features)} days')
print(f'Model test RMSE: {model_artifact["test_rmse"]:.1f} MW')
print(f'Model test R²: {model_artifact["test_r2"]:.4f}')

## Operational Simulation (2024 Base Period)

For each day in the test period, restrict information to what would be
available at 7 AM and generate RED/YELLOW/GREEN alerts.

In [ ]:
def generate_alerts(model, features_df, feature_cols, buffer_mw=500):
    """Simulate operational prediction for a set of days."""
    df = features_df.copy()
    X = df[feature_cols].copy()
    
    # Handle missing features
    complete_mask = X.notna().all(axis=1)
    df = df[complete_mask].copy()
    X = X[complete_mask].copy()
    
    # Predict
    y_pred = model.predict(X)
    df['predicted_max_demand'] = y_pred
    
    # Generate alerts
    alerts = []
    for idx, row in df.iterrows():
        threshold = row['current_5th_peak'] if row['current_5th_peak'] > 0 else 20000
        diff = row['predicted_max_demand'] - threshold
        if diff > buffer_mw:
            alerts.append('RED')
        elif abs(diff) <= buffer_mw:
            alerts.append('YELLOW')
        else:
            alerts.append('GREEN')
    
    df['alert'] = alerts
    df['is_alert'] = df['alert'].isin(['RED', 'YELLOW']).astype(int)
    
    return df

# Run simulation on 2024 base period
test_data = features[features['base_period'] == 2024].copy()
test_results = generate_alerts(model_artifact['model'], test_data, FEATURE_COLS)

print(f'Test period: {test_results["Date"].min()} to {test_results["Date"].max()}')
print(f'Total days: {len(test_results)}')
print(f'\nAlert distribution:')
print(test_results['alert'].value_counts().to_string())
print(f'\nActual peak days: {test_results["is_peak_day"].sum()}')
print(f'Alert days: {test_results["is_alert"].sum()}')

## Confusion Matrix

In [ ]:
# Confusion matrix for peak day detection
cm = confusion_matrix(test_results['is_peak_day'], test_results['is_alert'])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Alert', 'Alert (RED/YELLOW)'],
            yticklabels=['Non-Peak Day', 'Peak Day'],
            annot_kws={'fontsize': 16})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Peak Day Detection — Confusion Matrix (2024 Base Period)', fontsize=13)

# Add metrics text
prec = precision_score(test_results['is_peak_day'], test_results['is_alert'], zero_division=0)
rec = recall_score(test_results['is_peak_day'], test_results['is_alert'], zero_division=0)
f1 = f1_score(test_results['is_peak_day'], test_results['is_alert'], zero_division=0)
ax.text(0.02, -0.15, f'Precision: {prec:.1%}  |  Recall: {rec:.1%}  |  F1: {f1:.3f}',
        transform=ax.transAxes, fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig(DATA_DIR / 'confusion_matrix_2024.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Precision: {prec:.1%} — {prec*100:.0f}% of alerts were actual peak days')
print(f'Recall: {rec:.1%} — {rec*100:.0f}% of actual peak days received alerts')
print(f'False alarms: {cm[0, 1]} days')
print(f'Missed peaks: {cm[1, 0]} days')

## Alert Calendar Visualization

In [ ]:
# Calendar heatmap of alerts with actual peaks marked
# Focus on summer months (June-August) when peaks concentrate
summer = test_results[
    (test_results['Date'].dt.month >= 5) & 
    (test_results['Date'].dt.month <= 9)
].copy()

alert_colors = {'GREEN': 0, 'YELLOW': 1, 'RED': 2}
summer['alert_code'] = summer['alert'].map(alert_colors)
summer['week'] = summer['Date'].dt.isocalendar().week.astype(int)
summer['dow'] = summer['Date'].dt.dayofweek

fig, ax = plt.subplots(figsize=(16, 6))

# Plot each day as a colored square
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#C8E6C9', '#FFF9C4', '#FFCDD2'])  # green, yellow, red

weeks = sorted(summer['week'].unique())
for _, row in summer.iterrows():
    week_idx = weeks.index(row['week'])
    color = ['#C8E6C9', '#FFF9C4', '#FFCDD2'][row['alert_code']]
    rect = plt.Rectangle((week_idx, row['dow']), 0.9, 0.9, 
                          facecolor=color, edgecolor='white', linewidth=1)
    ax.add_patch(rect)
    
    # Mark actual peak days with a star
    if row['is_peak_day'] == 1:
        ax.plot(week_idx + 0.45, row['dow'] + 0.45, '*', 
                color='black', markersize=15, zorder=5)
    
    # Add date label
    ax.text(week_idx + 0.45, row['dow'] + 0.45, row['Date'].strftime('%d'),
            ha='center', va='center', fontsize=6, color='#555')

ax.set_xlim(-0.5, len(weeks) + 0.5)
ax.set_ylim(-0.5, 7)
ax.set_yticks(np.arange(7) + 0.45)
ax.set_yticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax.set_xlabel('Week')
ax.set_title('Alert Calendar — 2024 Base Period (May–Sep)\n'
             'Stars mark actual top-5 peak days', fontsize=13)
ax.invert_yaxis()

# Legend
legend_patches = [
    mpatches.Patch(facecolor='#C8E6C9', edgecolor='gray', label='GREEN'),
    mpatches.Patch(facecolor='#FFF9C4', edgecolor='gray', label='YELLOW'),
    mpatches.Patch(facecolor='#FFCDD2', edgecolor='gray', label='RED'),
    plt.Line2D([0], [0], marker='*', color='black', linestyle='None', 
               markersize=12, label='Actual Peak'),
]
ax.legend(handles=legend_patches, loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.savefig(DATA_DIR / 'alert_calendar_2024.png', dpi=150, bbox_inches='tight')
plt.show()

## 3-Hour Window Evaluation

For correctly predicted peak days, determine how often the actual peak hour
fell within the predicted 3-hour window.

In [ ]:
# Load hourly data for window evaluation
hourly = pd.read_parquet(DATA_DIR / 'ieso_hourly_with_features.parquet')
hourly['Date'] = pd.to_datetime(hourly['Date'])

# Get the actual peak hours from ground truth
top5 = peaks[peaks['rank'] <= 5].copy()
bp2024_peaks = top5[top5['base_period'] == 2024]

# For each predicted peak day, identify the predicted 3-hour window
# based on historical peak hour distribution and temperature profile
def predict_peak_window(date, hourly_data):
    """Predict the most likely 3-hour peak window based on temperature profile.
    
    Hotter afternoons push peaks later (sustained AC load into evening).
    Base window: HE 16-18 (3-6 PM). Shift later for high temps.
    """
    day_data = hourly_data[hourly_data['Date'] == date]
    if len(day_data) == 0:
        return [16, 17, 18]  # Default window
    
    max_temp = day_data['temperature_c'].max()
    
    if max_temp > 33:
        return [17, 18, 19]  # Very hot: peaks later
    elif max_temp > 28:
        return [16, 17, 18]  # Hot: standard window
    else:
        return [15, 16, 17]  # Moderate: peaks earlier

# Evaluate window accuracy on correctly predicted peak days
correctly_alerted = test_results[
    (test_results['is_peak_day'] == 1) & (test_results['is_alert'] == 1)
]

window_results = []
for _, row in correctly_alerted.iterrows():
    predicted_window = predict_peak_window(row['Date'], hourly)
    actual_peak = bp2024_peaks[bp2024_peaks['date'].dt.date == row['Date'].date()]
    if len(actual_peak) > 0:
        actual_hour = actual_peak['hour_ending'].values[0]
        in_window = actual_hour in predicted_window
        window_results.append({
            'date': row['Date'],
            'actual_hour': actual_hour,
            'predicted_window': f'HE {predicted_window[0]}-{predicted_window[-1]}',
            'in_window': in_window
        })

if window_results:
    window_df = pd.DataFrame(window_results)
    window_accuracy = window_df['in_window'].mean()
    print(f'3-Hour Window Accuracy: {window_accuracy:.1%}')
    print(f'\nWindow evaluation details:')
    print(window_df.to_string(index=False))
else:
    print('No correctly alerted peak days to evaluate window accuracy.')
    window_accuracy = 0.0

## Walk-Forward Backtest (2019–2024)

Retrain model using only data available prior to each base period,
then predict that period's peaks. This simulates true operational conditions.

In [ ]:
def walk_forward_backtest(features_df, feature_cols, test_periods):
    """Walk-forward validation: retrain before each test period."""
    results = []
    
    for bp in test_periods:
        # Train on all base periods before this one
        train_mask = (features_df['base_period'] < bp) & \
                     features_df[feature_cols + ['daily_max_demand']].notna().all(axis=1)
        test_mask = (features_df['base_period'] == bp) & \
                    features_df[feature_cols + ['daily_max_demand']].notna().all(axis=1)
        
        train_data = features_df[train_mask]
        test_data = features_df[test_mask]
        
        if len(train_data) < 100 or len(test_data) < 30:
            continue
        
        X_tr = train_data[feature_cols]
        y_tr = train_data['daily_max_demand']
        X_te = test_data[feature_cols]
        y_te = test_data['daily_max_demand']
        
        # Train XGBoost
        model = xgb.XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
            random_state=42, verbosity=0
        )
        model.fit(X_tr, y_tr)
        
        # Generate alerts
        test_with_alerts = generate_alerts(model, test_data, feature_cols)
        
        # Metrics
        y_pred = model.predict(X_te)
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        prec = precision_score(test_with_alerts['is_peak_day'], 
                              test_with_alerts['is_alert'], zero_division=0)
        rec = recall_score(test_with_alerts['is_peak_day'], 
                          test_with_alerts['is_alert'], zero_division=0)
        
        results.append({
            'base_period': bp,
            'train_years': f'2010-{bp-1}',
            'train_days': len(train_data),
            'test_days': len(test_data),
            'peak_days': test_with_alerts['is_peak_day'].sum(),
            'alert_days': test_with_alerts['is_alert'].sum(),
            'rmse_mw': rmse,
            'precision': prec,
            'recall': rec,
            'f1': f1_score(test_with_alerts['is_peak_day'], 
                          test_with_alerts['is_alert'], zero_division=0),
        })
        print(f'  BP {bp}: RMSE={rmse:.0f} MW, Precision={prec:.1%}, '
              f'Recall={rec:.1%}, Alerts={test_with_alerts["is_alert"].sum()}')
    
    return pd.DataFrame(results)

print('Walk-forward backtest (2019-2024)...')
backtest_results = walk_forward_backtest(
    features, FEATURE_COLS, 
    test_periods=[2019, 2020, 2021, 2022, 2023, 2024]
)

print(f'\n=== Walk-Forward Results ===')
print(backtest_results.round(3).to_string(index=False))

In [ ]:
# Walk-forward performance chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# RMSE over time
axes[0].bar(backtest_results['base_period'].astype(str), 
            backtest_results['rmse_mw'], color='#1565C0', alpha=0.8)
axes[0].set_xlabel('Base Period')
axes[0].set_ylabel('RMSE (MW)')
axes[0].set_title('Prediction Error')
axes[0].tick_params(axis='x', rotation=45)

# Precision and Recall over time
x = np.arange(len(backtest_results))
axes[1].bar(x - 0.2, backtest_results['precision'], 0.4, 
            label='Precision', color='#4CAF50', alpha=0.8)
axes[1].bar(x + 0.2, backtest_results['recall'], 0.4, 
            label='Recall', color='#FF9800', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(backtest_results['base_period'].astype(str), rotation=45)
axes[1].set_xlabel('Base Period')
axes[1].set_ylabel('Score')
axes[1].set_title('Peak Detection Performance')
axes[1].legend()
axes[1].set_ylim(0, 1.1)

# Alert days vs peak days
axes[2].bar(x - 0.2, backtest_results['alert_days'], 0.4, 
            label='Alert Days', color='#d32f2f', alpha=0.8)
axes[2].bar(x + 0.2, backtest_results['peak_days'], 0.4, 
            label='Actual Peak Days', color='#1565C0', alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(backtest_results['base_period'].astype(str), rotation=45)
axes[2].set_xlabel('Base Period')
axes[2].set_ylabel('Days')
axes[2].set_title('Alert Volume')
axes[2].legend()

plt.suptitle('Walk-Forward Backtest Performance (2019–2024)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'walkforward_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## Financial Valuation

Quantify the dollar value of the prediction model for a hypothetical
10 MW Class A customer.

In [ ]:
# Financial parameters
CUSTOMER_BASELINE_MW = 10.0    # Normal operating demand
CURTAILMENT_MW = 2.0           # Demand during curtailment
GA_RATE_PER_MWH = 40.0         # Average GA rate ($/MWh)
MONTHLY_GA_TOTAL = 1.5e9 / 12  # ~$125M/month total system GA
SYSTEM_PEAK_MW = 22000.0       # Approximate system-wide peak demand
CURTAILMENT_COST_PER_EVENT = 15000  # Lost production per curtailment day ($)
CURTAILMENT_HOURS = 3          # Hours of curtailment per alert day

def compute_pdf(customer_mw, system_mw, n_peaks=5):
    """Compute Peak Demand Factor.
    PDF = sum(customer MWh during peaks) / sum(system MWh during peaks)
    """
    customer_total = customer_mw * n_peaks  # MWh (1 hour each)
    system_total = system_mw * n_peaks
    return customer_total / system_total

def annual_ga_cost(pdf, monthly_ga=MONTHLY_GA_TOTAL):
    """Annual GA cost = PDF * monthly GA * 12."""
    return pdf * monthly_ga * 12

# Scenario A: No curtailment
pdf_no_curtail = compute_pdf(CUSTOMER_BASELINE_MW, SYSTEM_PEAK_MW)
cost_no_curtail = annual_ga_cost(pdf_no_curtail)

# Scenario C: Perfect foresight (curtail during only actual 5 peaks)
pdf_perfect = compute_pdf(CURTAILMENT_MW, SYSTEM_PEAK_MW)
cost_perfect = annual_ga_cost(pdf_perfect)
savings_perfect = cost_no_curtail - cost_perfect
curtailment_cost_perfect = 5 * CURTAILMENT_COST_PER_EVENT
net_value_perfect = savings_perfect - curtailment_cost_perfect

print('=== Financial Parameters ===')
print(f'Customer baseline demand: {CUSTOMER_BASELINE_MW} MW')
print(f'Curtailment level: {CURTAILMENT_MW} MW')
print(f'PDF (no curtailment): {pdf_no_curtail:.6f}')
print(f'PDF (perfect curtailment): {pdf_perfect:.6f}')
print(f'\nAnnual GA cost (no curtailment): ${cost_no_curtail:,.0f}')
print(f'Annual GA cost (perfect foresight): ${cost_perfect:,.0f}')
print(f'Maximum possible savings: ${savings_perfect:,.0f}')

In [ ]:
# Compute financial outcomes for each scenario across backtest periods
def evaluate_financial(test_results_df, customer_mw=CUSTOMER_BASELINE_MW,
                      curtail_mw=CURTAILMENT_MW, system_mw=SYSTEM_PEAK_MW,
                      curtail_cost=CURTAILMENT_COST_PER_EVENT):
    """Evaluate financial impact of the model's alerts."""
    df = test_results_df.copy()
    
    # Count outcomes
    true_positives = ((df['is_peak_day'] == 1) & (df['is_alert'] == 1)).sum()
    false_positives = ((df['is_peak_day'] == 0) & (df['is_alert'] == 1)).sum()
    false_negatives = ((df['is_peak_day'] == 1) & (df['is_alert'] == 0)).sum()
    total_peak_days = df['is_peak_day'].sum()
    total_alert_days = df['is_alert'].sum()
    
    # Customer's effective demand during peaks:
    # - If alerted AND it was a peak: customer at curtailment level
    # - If NOT alerted AND it was a peak: customer at baseline
    peaks_curtailed = true_positives
    peaks_missed = false_negatives
    
    # Effective customer MWh during peaks
    customer_peak_mwh = (peaks_curtailed * curtail_mw + 
                         peaks_missed * customer_mw)
    system_peak_mwh = total_peak_days * system_mw
    
    pdf = customer_peak_mwh / system_peak_mwh if system_peak_mwh > 0 else 0
    ga_cost = annual_ga_cost(pdf)
    ga_savings = cost_no_curtail - ga_cost
    total_curtail_cost = total_alert_days * curtail_cost
    net_value = ga_savings - total_curtail_cost
    
    return {
        'peaks_curtailed': peaks_curtailed,
        'peaks_missed': peaks_missed,
        'false_alarms': false_positives,
        'total_alert_days': total_alert_days,
        'pdf': pdf,
        'ga_cost': ga_cost,
        'ga_savings': ga_savings,
        'curtailment_cost': total_curtail_cost,
        'net_value': net_value,
    }

# Scenario B: Model-guided curtailment
model_financial = evaluate_financial(test_results)

# Scenario D: Naive heuristic
test_naive = test_results.copy()
test_naive['is_alert'] = (
    (test_naive['daily_max_temp'] > 30) & 
    (test_naive['is_business_day'] == 1) & 
    (test_naive['month'].isin([6, 7, 8]))
).astype(int)
naive_financial = evaluate_financial(test_naive)

# Summary table
scenarios = pd.DataFrame([
    {
        'Scenario': 'A: No Curtailment',
        'Alert Days': 0,
        'Peaks Caught': 0,
        'Peaks Missed': 5,
        'False Alarms': 0,
        'GA Cost': cost_no_curtail,
        'GA Savings': 0,
        'Curtailment Cost': 0,
        'Net Value': 0,
    },
    {
        'Scenario': 'B: Model-Guided',
        'Alert Days': model_financial['total_alert_days'],
        'Peaks Caught': model_financial['peaks_curtailed'],
        'Peaks Missed': model_financial['peaks_missed'],
        'False Alarms': model_financial['false_alarms'],
        'GA Cost': model_financial['ga_cost'],
        'GA Savings': model_financial['ga_savings'],
        'Curtailment Cost': model_financial['curtailment_cost'],
        'Net Value': model_financial['net_value'],
    },
    {
        'Scenario': 'C: Perfect Foresight',
        'Alert Days': 5,
        'Peaks Caught': 5,
        'Peaks Missed': 0,
        'False Alarms': 0,
        'GA Cost': cost_perfect,
        'GA Savings': savings_perfect,
        'Curtailment Cost': curtailment_cost_perfect,
        'Net Value': net_value_perfect,
    },
    {
        'Scenario': 'D: Naive Heuristic',
        'Alert Days': naive_financial['total_alert_days'],
        'Peaks Caught': naive_financial['peaks_curtailed'],
        'Peaks Missed': naive_financial['peaks_missed'],
        'False Alarms': naive_financial['false_alarms'],
        'GA Cost': naive_financial['ga_cost'],
        'GA Savings': naive_financial['ga_savings'],
        'Curtailment Cost': naive_financial['curtailment_cost'],
        'Net Value': naive_financial['net_value'],
    },
])

# Format for display
print('=== Financial Valuation — 10 MW Class A Customer (2024 Base Period) ===')
display_df = scenarios.copy()
for col in ['GA Cost', 'GA Savings', 'Curtailment Cost', 'Net Value']:
    display_df[col] = display_df[col].apply(lambda x: f'${x:,.0f}')
print(display_df.to_string(index=False))

In [ ]:
# Financial comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Net value comparison
scenario_names = scenarios['Scenario'].values
net_values = scenarios['Net Value'].values
colors = ['#9E9E9E', '#4CAF50', '#1565C0', '#FF9800']
axes[0].bar(range(4), net_values, color=colors)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(['No Curtail', 'Model', 'Perfect', 'Naive'], rotation=0)
axes[0].set_ylabel('Net Value ($)')
axes[0].set_title('Annual Net Value by Strategy')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Alert days breakdown
x_pos = range(4)
axes[1].bar(x_pos, scenarios['Alert Days'].values, color=colors, alpha=0.7)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(['No Curtail', 'Model', 'Perfect', 'Naive'], rotation=0)
axes[1].set_ylabel('Days')
axes[1].set_title('Curtailment Days Required')

# Annotate with peaks caught
for i, (_, row) in enumerate(scenarios.iterrows()):
    axes[1].text(i, row['Alert Days'] + 0.5, 
                f"{row['Peaks Caught']}/5", ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Financial Valuation — Model vs. Alternative Strategies', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'financial_valuation.png', dpi=150, bbox_inches='tight')
plt.show()

# Value proposition summary
model_vs_none = model_financial['net_value']
model_vs_naive = model_financial['net_value'] - naive_financial['net_value']
print(f'\n=== Value Proposition ===')
print(f'Model net value vs. no curtailment: ${model_vs_none:,.0f}')
print(f'Model net value vs. naive heuristic: ${model_vs_naive:,.0f}')
print(f'False alarm days per year: {model_financial["false_alarms"]}')
print(f'Peaks caught: {model_financial["peaks_curtailed"]}/5')

# Save results
scenarios.to_csv(DATA_DIR / 'financial_valuation.csv', index=False)
backtest_results.to_csv(DATA_DIR / 'walkforward_results.csv', index=False)

print('\n=== Notebook 4 complete ===')